# MSI DTi2vec: local seed-0 reconstruction

Select a local **Python 3 kernel → Run All**. First use creates a separate environment with network access. Computation runs in subprocesses without switching the host kernel.

Pipeline: `Target similarity + semantic similarity → KNN graph → node2vec / Skip-gram → XGBoost`.

Reuse prepared MSI, MeSH/ATC and seed-0 negatives. Select K by **three-fold inner CV within each of ten outer folds**. New outputs go to `02.DTi2vec_reproduction/`. Training starts when you run the cells; published outputs are cleared.


## 1. Directories and run name

Normally no changes are needed. Set `REPOSITORY_OVERRIDE` if the notebook server starts outside this repository; `PYTHON_OVERRIDE` may select an interpreter satisfying the package pins. Completed fits are reused. Change `RUN_NAME` after changing inputs, settings or source.


In [ ]:
from pathlib import Path
import csv
import html
import json
import os
import shutil
import signal
import subprocess
import sys
from IPython.display import display, HTML

RUN_NAME = 'msi_seed0'
REPOSITORY_OVERRIDE = None
PYTHON_OVERRIDE = None

def find_reproduction_root(start):
    start = Path(start).expanduser().resolve()
    for parent in [start, *start.parents]:
        for candidate in [parent / '00.Reproduction', parent]:
            if ((candidate / '00.DREAMwalk_reproduction').is_dir()
                    and (candidate / '02.DTi2vec_reproduction').is_dir()):
                return candidate
    raise FileNotFoundError('00.Reproduction not found; set REPOSITORY_OVERRIDE to the repository root or 00.Reproduction.')

REPOSITORY = find_reproduction_root(REPOSITORY_OVERRIDE or Path.cwd())
ROOT = REPOSITORY / '02.DTi2vec_reproduction'
SOURCE = REPOSITORY / '00.DREAMwalk_reproduction'
ENV_PREFIX = ROOT / 'environment/.conda-env'
PYTHON = Path(PYTHON_OVERRIDE).expanduser().resolve() if PYTHON_OVERRIDE else ENV_PREFIX / 'bin/python'
RUNNER = ROOT / 'src/run_relocated_dti2vec.py'
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('RUN_NAME accepts only letters, digits, underscores and hyphens.')
RESULTS = ROOT / 'results' / RUN_NAME
LOGS = ROOT / 'logs' / RUN_NAME
LOGS.mkdir(parents=True, exist_ok=True)
for path in [SOURCE / 'data/processed/msi_graph.tsv',
             SOURCE / 'data/processed/msi_nodetypes.tsv',
             SOURCE / 'data/processed/msi_positive_pairs.tsv',
             SOURCE / 'data/processed/msi_semantic_hierarchy_final.csv',
             SOURCE / 'results/msi_repeat0_negative_pairs.csv', RUNNER]:
    if not path.is_file():
        raise FileNotFoundError(f'Required input or runner missing: {path}')
print('Project: ', ROOT)
print('Computation interpreter: ', PYTHON)
print('Results: ', RESULTS)


## 2. Independent environment: installed on first run

Create `environment/.conda-env/` with local mamba/conda. Python 3.8, NumPy, SciPy, scikit-learn and XGBoost follow DREAMwalk pins; Gensim 4.3.2 provides standard Skip-gram. The current kernel only starts subprocesses and displays results. Save installation output to `logs/<RUN_NAME>/environment.log`.


In [ ]:
def stream_command(command, log_path, env=None):
    command = list(map(str, command))
    print('Starting: ', ' '.join(command), flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\nCOMMAND: ' + repr(command) + '\n')
        with subprocess.Popen(command, cwd=ROOT, env=env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, start_new_session=True) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
                raise
            if returncode:
                raise subprocess.CalledProcessError(returncode, command)

PINS = {'numpy': '1.23.3', 'scipy': '1.9.1', 'scikit-learn': '1.1.3',
        'networkx': '2.8.8', 'pandas': '1.4.4', 'gensim': '4.3.2',
        'smart_open': '6.4.0', 'xgboost': '1.7.4'}
probe = (
    'import sys; from importlib.metadata import version; '
    'import numpy, scipy, sklearn, networkx, pandas, gensim, xgboost; '
    'assert sys.version_info[:2] == (3,8); '
    f'expected={PINS!r}; '
    'actual={k:version(k) for k in expected}; '
    'assert actual == expected, actual; print(sys.version); print(actual)'
)
ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True) if PYTHON.is_file() else None
if ready is None or ready.returncode:
    if PYTHON_OVERRIDE:
        if ready is not None:
            print(ready.stdout, ready.stderr)
        raise RuntimeError('PYTHON_OVERRIDE does not match the environment; clear it to install a dedicated environment.')
    candidates = [shutil.which('mamba'), shutil.which('conda'),
                  str(Path.home() / 'mambaforge/bin/mamba'),
                  str(Path.home() / 'miniforge3/bin/mamba'),
                  str(Path.home() / 'miniforge3/bin/conda'),
                  str(Path.home() / 'miniconda3/bin/conda')]
    manager = next((p for p in candidates if p and Path(p).is_file()), None)
    if manager is None:
        raise RuntimeError('mamba/conda not found; install Miniforge and rerun this section.')
    action = 'install' if (ENV_PREFIX / 'conda-meta/history').is_file() else 'create'
    command = [manager, action, '--yes', '--prefix', str(ENV_PREFIX),
               '--override-channels', '-c', 'conda-forge', 'python=3.8']
    command += [f'{package}={version}' for package, version in PINS.items()]
    install_env = dict(os.environ)
    install_env['CONDA_PKGS_DIRS'] = str(ROOT / 'environment/.conda-pkgs')
    stream_command(command, LOGS / 'environment.log', env=install_env)
    ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True)
    if ready.returncode:
        print(ready.stdout, ready.stderr)
        raise RuntimeError('Environment is not ready; inspect the installation log above.')
print(ready.stdout)


## 3. Fixed configuration

| Component | Setting |
|---|---|
| Target similarity | MSI gene-set Jaccard for drug–drug and disease–disease pairs |
| Semantic similarity | Recalculate complete similarities with shared MeSH/ATC mappings, hierarchies and upstream code |
| Fusion / KNN | `(Jaccard + semantic) / 2`; missing semantic=0; top-K positive neighbors; undirected union |
| Embedding graph | 2,501 drug/disease nodes; add only the current training split's treatment edges, weight=1 |
| K selection | {5,10,20,50}; mean three-fold inner AUPR; prefer smaller K on ties |
| Walks / Skip-gram | p=q=1; 100 walks/node; length 10; 128D; window 4; one epoch; negative=5; four workers |
| Predictor | 256D concatenation → training-fit MinMaxScaler → XGBoost |
| Evaluation | Same 5,926 positives + 5,926 seed-0 negatives; ten outer folds |

KNN builds the graph; node2vec generates walks. Standard Gensim Skip-gram has no DREAMwalk `pp`, teleport `τ` or edge-type EM.

**Reconstruction choices:** the MSI-specific K and complete benchmark script are not released. Record arithmetic-mean fusion, KNN union, K grid, inner CV and training-DDA bridge construction explicitly. Do not choose K by closeness to the paper score.


In [ ]:
CHILD_ENV = dict(os.environ)
CHILD_ENV['PYTHONUNBUFFERED'] = '1'
CHILD_ENV['PYTHONDONTWRITEBYTECODE'] = '1'

def run_phase(phase):
    stream_command([PYTHON, '-u', RUNNER, phase, '--run-name', RUN_NAME],
                   LOGS / f'{phase}.log', env=CHILD_ENV)

run_phase('config')


## 4. Prepare target/semantic similarities and KNN graphs

**Computation starts here.** Compute target similarity from prepared molecular associations. Semantic similarity uses shared MeSH/ATC sources but recalculates the full matrix rather than reusing DREAMwalk's cutoff-0.4 teleport graph.

Save similarities and KNN graphs under `data/<RUN_NAME>/`. Add treatment edges separately within each fit's training partition, keeping validation/test positives outside that embedding graph.


In [ ]:
run_phase('prepare')


## 5. Inner K selection and ten-fold outer XGBoost

For each outer fold:

1. Run three-fold inner CV on outer training data, with four K values per inner fold.
2. Select K by mean inner AUPR.
3. Retrain embeddings and XGBoost on outer training data; evaluate the outer test fold once.

This requires **130 embedding/classifier fits**, each with **250,100 length-10 walks** for 2,501 nodes. Logs report progress per fit.

**Save every completed embedding immediately in its fold/K directory.** Completed fits resume automatically; incomplete embedding fits must restart.


In [ ]:
run_phase('benchmark')
print('Completed all ten outer folds and their inner K selections.')


## 6. Compare with the paper's DTi2vec baseline

Use **DTi2vec's own** Figure 2a Source Data reference. Local evaluation is one outer ten-fold repeat; the paper uses ten repeats. Inspect selected K, outer-fold results and means. No additional seeds or K candidates are started automatically.


In [ ]:
run_phase('compare')

def show_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        columns = reader.fieldnames
        rows = list(reader)
    if not columns:
        print(path.name, ': empty table')
        return
    def pretty(value):
        try:
            return f'{float(value):.6f}'
        except (TypeError, ValueError):
            return str(value)
    header = ''.join(f'<th>{html.escape(c)}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(pretty(row[c]))}</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table>'))

for filename in ['selected_k.csv', 'fold_results.csv', 'summary.csv', 'paper_comparison.csv']:
    print(filename)
    show_csv(RESULTS / filename)
print('Completed DTi2vec seed-0 outer ten-fold evaluation.')
print('Results: ', RESULTS)
print('Embedding checkpoints：', ROOT / 'data' / RUN_NAME)


## Outputs and sources

- Similarities: `data/<RUN_NAME>/similarities.npz`; KNN graphs: `graph_knn_k<K>.npz`.
- Embeddings: e.g. `outer01/inner01/k005/embedding.pkl` and `outer01/final/k005/embedding.pkl` under the run data directory.
- Results: `inner_cv_scores.csv`, `selected_k.csv`, `fold_results.csv`, `fold_assignments.csv`, `oof_predictions.csv`, `summary.csv`, `paper_comparison.csv`.
- Configuration/completion JSON: `results/<RUN_NAME>/`; logs: `logs/<RUN_NAME>/`.
- Four-thread Skip-gram may vary numerically. Record the inherited startup hash seed without overriding it.

[Bang et al. (2023)](https://www.nature.com/articles/s41467-023-39301-y) · [Supplementary Method 1](https://media.springernature.com/original/springer-static/esm/art%3A10.1038%2Fs41467-023-39301-y/MediaObjects/41467_2023_39301_MOESM1_ESM.pdf) · [DTi2vec paper](https://doi.org/10.1186/s13321-021-00552-w) · [DTi2vec reference](https://github.com/MahaThafar/DTi2Vec)
